In [ ]:
from __future__ import annotations

import jax
import jax.numpy as jnp
import equinox as eqx
import numpy as np
import optax
from diffrax import Dopri5, ODETerm, PIDController, SaveAt, diffeqsolve
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

from mxlpy import Model
from mxlpy.jax import to_jax_export

# Universal Differential Equations for Plant Fluorescence

This notebook demonstrates a complete **UDE** (Universal Differential Equation) workflow
using mxlpy's JAX export.

**Scenario:**  A small mechanistic model captures the quenching dynamics of plant
chlorophyll fluorescence under a changing illumination protocol.  The *true* biological
system contains an unknown periodic forcing — simulating e.g. stomatal oscillations or
circadian rhythm crosstalk — that the mechanistic model does not account for.

A Neural Differential Equation (NDE) is appended as an additive correction to the
mechanistic RHS.  After training on trajectory data the NDE recovers the hidden periodic
term.

```
dQ/dt = k_ind·I − k_rel·Q         ← mechanistic  (JaxExport.rhs)
       + nn([t̄, Q̄, Ī])            ← unknown forcing  (Equinox MLP, to be learned)
```

## 1 · Mechanistic fluorescence model

The quenching state $Q$ builds up under illumination and relaxes spontaneously:

$$\frac{dQ}{dt} = k_\text{ind}\cdot I - k_\text{rel}\cdot Q$$

Fluorescence yield follows the Stern–Volmer relationship:

$$F(t) = \frac{1}{1 + Q(t)}$$

where $I$ is the photon flux density (PFD, μmol m⁻² s⁻¹).

In [ ]:
def linear_induction(k_ind: float, I: float) -> float:
    """Quenching induction proportional to light intensity."""
    return k_ind * I


def mass_action_relaxation(k_rel: float, Q: float) -> float:
    """First-order quenching relaxation."""
    return k_rel * Q


model = (
    Model()
    .add_variable("Q", 0.0)
    .add_parameters({"k_ind": 0.001, "k_rel": 0.1, "I": 100.0})
    .add_reaction(
        "induction",
        linear_induction,
        stoichiometry={"Q": 1},
        args=["k_ind", "I"],
    )
    .add_reaction(
        "relaxation",
        mass_action_relaxation,
        stoichiometry={"Q": -1},
        args=["k_rel", "Q"],
    )
)

print("Variables :", model.get_variable_names())
print("Parameters:", list(model.get_parameter_values()))

## 2 · Illumination protocol

Three phases of 100 s each:

| Phase | Duration | PFD |
|-------|----------|-----|
| Low light   | 0 – 100 s   | 100 PFD  |
| High light  | 100 – 200 s | 2000 PFD |
| Medium light| 200 – 300 s | 500 PFD  |

In [ ]:
T_TOTAL = 300.0  # total simulation time (s)
PHASE = 100.0  # duration of each light phase (s)
LEVELS = (100.0, 2000.0, 500.0)  # PFD for each phase


def light_np(t: float | np.ndarray) -> float | np.ndarray:
    """Piecewise constant light protocol — NumPy version (for data generation)."""
    t = np.asarray(t)
    return np.where(t < PHASE, LEVELS[0], np.where(t < 2 * PHASE, LEVELS[1], LEVELS[2]))


def light_jax(t: float) -> float:
    """Piecewise constant light protocol — JAX version (for UDE training)."""
    return jnp.where(
        t < PHASE, LEVELS[0], jnp.where(t < 2 * PHASE, LEVELS[1], LEVELS[2])
    )


t_plot = np.linspace(0, T_TOTAL, 1000)
fig, ax = plt.subplots(figsize=(9, 2.5))
ax.step(t_plot, light_np(t_plot), where="post", color="goldenrod", linewidth=2.5)
ax.set(xlabel="Time (s)", ylabel="PFD (μmol m⁻² s⁻¹)", title="Illumination protocol")
ax.set_ylim(0, 2400)
plt.tight_layout()

## 3 · Synthetic data — true model with hidden sinusoidal forcing

The *true* system includes a periodic term absent from the mechanistic model:

$$\frac{dQ}{dt} = k_\text{ind}\cdot I - k_\text{rel}\cdot Q
   + \underbrace{A\sin(\omega t)}_{\text{hidden forcing}}$$

We integrate this to generate trajectory data that the UDE will be trained on.

In [ ]:
# Hidden parameters — unknown to the mechanistic model
A_TRUE = 0.3  # amplitude (quenching units)
OMEGA_TRUE = 2 * np.pi / 20.0  # angular frequency (period ≈ 20 s)

K_IND = model.get_parameter_values()["k_ind"]
K_REL = model.get_parameter_values()["k_rel"]


def true_rhs(t: float, y: np.ndarray) -> list[float]:
    """True ODE: mechanistic + hidden sinusoidal forcing."""
    Q = y[0]
    I = float(light_np(t))
    return [K_IND * I - K_REL * Q + A_TRUE * np.sin(OMEGA_TRUE * t)]


t_data = np.linspace(0.0, T_TOTAL, 301)
sol = solve_ivp(
    true_rhs,
    [0.0, T_TOTAL],
    [0.0],
    t_eval=t_data,
    method="RK45",
    rtol=1e-7,
    atol=1e-9,
)
Q_true = sol.y[0]
F_true = 1.0 / (1.0 + Q_true)

print(f"Q range : [{Q_true.min():.2f}, {Q_true.max():.2f}]")
print(f"F range : [{F_true.min():.3f}, {F_true.max():.3f}]")

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
ax1.step(t_data, light_np(t_data), where="post", color="goldenrod", lw=2)
ax1.set(ylabel="PFD", title="Illumination")
ax2.plot(t_data, Q_true, color="steelblue", lw=1.5)
ax2.set(ylabel="Q", title="Quenching state (true, with hidden forcing)")
ax3.plot(t_data, F_true, color="forestgreen", lw=1.5)
ax3.set(xlabel="Time (s)", ylabel="F", title="Fluorescence yield (true)")
plt.tight_layout()

## 4 · Export the mechanistic model to JAX

`to_jax_export` converts the mxlpy model to a differentiable RHS function via its
SymPy symbolic representation.  The returned `JaxExport` object is compatible with
Diffrax and supports `jax.jit`, `jax.grad`, and `jax.vmap`.

In [ ]:
export = to_jax_export(model)

print("variable_names :", export.variable_names)
print("parameter_names:", export.parameter_names)

# Sanity check: exported RHS matches model() at t=0
y0_jax = export.get_y0()
args0 = export.get_args()
dydt_jax = export.rhs(0.0, y0_jax, args0)
dydt_model = model(0.0, list(y0_jax))
print(f"\nRHS at t=0, Q=0:  JAX={float(dydt_jax[0]):.5f}  model={dydt_model[0]:.5f}")

# Index of the light intensity parameter (needed to override it in the UDE)
I_IDX = export.parameter_names.index("I")
print(f"'I' is at index {I_IDX} in parameter vector")

### Mechanistic simulation (no neural correction)

We first simulate the exported model **without** any neural correction to see the
residual that the NDE will need to explain.

In [ ]:
# Override I=100 → I=2000 → I=500 at the correct times by simulating each phase
# separately, or use the simulate helper with manually updated args.
# Here we use scipy as reference to show the mechanistic-only trajectory.


def mech_rhs(t: float, y: np.ndarray) -> list[float]:
    """Pure mechanistic model (no sinusoidal term)."""
    Q = y[0]
    I = float(light_np(t))
    return [K_IND * I - K_REL * Q]


sol_mech = solve_ivp(
    mech_rhs,
    [0.0, T_TOTAL],
    [0.0],
    t_eval=t_data,
    method="RK45",
    rtol=1e-7,
    atol=1e-9,
)
Q_mech = sol_mech.y[0]
F_mech = 1.0 / (1.0 + Q_mech)

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(t_data, Q_true, "k-", lw=1.5, label="True (with forcing)")
axes[0].plot(t_data, Q_mech, "b--", lw=1.5, label="Mechanistic only")
axes[0].set(ylabel="Q", title="Quenching — true vs. mechanistic")
axes[0].legend()
axes[1].plot(t_data, F_true, "k-", lw=1.5, label="True")
axes[1].plot(t_data, F_mech, "b--", lw=1.5, label="Mechanistic only")
axes[1].set(xlabel="Time (s)", ylabel="F", title="Fluorescence — true vs. mechanistic")
axes[1].legend()
plt.tight_layout()

## 5 · Building the Universal Differential Equation

The UDE augments the mechanistic RHS with a small MLP:

$$\frac{dQ}{dt} = \underbrace{k_\text{ind}\cdot I - k_\text{rel}\cdot Q}_{\text{mechanistic}}
+\; \underbrace{\text{MLP}(\bar{t},\, \bar{Q},\, \bar{I})}_{\text{neural correction}}$$

Inputs to the MLP are normalised to $[0,1]$:
$\bar{t} = t/T$,\; $\bar{Q} = Q/Q_\max$,\; $\bar{I} = I/I_\max$.

The light intensity $I(t)$ is injected into the parameter vector at each evaluation
step via a functional array update, keeping the exported RHS fully JAX-traceable.

In [ ]:
# Normalisation constants
Q_MAX = float(Q_true.max()) * 1.5  # headroom above training range
I_MAX = max(LEVELS)

# Frozen mechanistic parameters — I will be overridden per time step
PARAMS = export.get_args()
Y0 = export.get_y0()  # Q(0) = 0


def ude_rhs(t: float, y: jax.Array, args: tuple) -> jax.Array:
    """UDE right-hand side: mechanistic + neural additive correction."""
    params, nn = args

    # Inject the current (time-varying) light intensity into the parameter vector
    I = light_jax(t)
    params_t = params.at[I_IDX].set(I)

    # Mechanistic contribution
    mech = export.rhs(t, y, params_t)

    # Neural correction: normalised inputs → scalar correction for dQ/dt
    nn_input = jnp.array([t / T_TOTAL, y[0] / Q_MAX, I / I_MAX])
    correction = nn(nn_input)  # shape (1,)

    return mech + correction


# Small MLP: 3 → 32 → 32 → 1 with tanh activations
key = jax.random.PRNGKey(0)
nn = eqx.nn.MLP(3, 1, width_size=32, depth=2, key=key, activation=jnp.tanh)

n_params = sum(p.size for p in jax.tree_util.tree_leaves(eqx.filter(nn, eqx.is_array)))
print(f"NDE parameters: {n_params}")

## 6 · Training

We minimise the mean-squared error between the UDE's predicted $Q$ trajectory and
the synthetic training data.  Gradients are computed through the Diffrax adaptive solver
using the continuous adjoint / discretise-then-optimise approach handled automatically
by `eqx.filter_value_and_grad`.

In [ ]:
T_TRAIN = jnp.array(t_data)  # save points
Q_TARGET = jnp.array(Q_true)  # training target


def simulate_ude(nn_model: eqx.nn.MLP) -> jax.Array:
    """Integrate the UDE over the full illumination protocol."""
    sol = diffeqsolve(
        ODETerm(ude_rhs),
        Dopri5(),
        t0=0.0,
        t1=T_TOTAL,
        dt0=0.5,
        y0=Y0,
        saveat=SaveAt(ts=T_TRAIN),
        stepsize_controller=PIDController(rtol=1e-3, atol=1e-5),
        args=(PARAMS, nn_model),
        max_steps=8192,
    )
    return sol.ys  # shape (n_steps, 1)


def compute_loss(nn_model: eqx.nn.MLP) -> jax.Array:
    """MSE between UDE prediction and training trajectory."""
    Q_pred = simulate_ude(nn_model)
    return jnp.mean((Q_pred[:, 0] - Q_TARGET) ** 2)


@eqx.filter_jit
def make_step(
    nn_model: eqx.nn.MLP,
    opt_state: optax.OptState,
) -> tuple[eqx.nn.MLP, optax.OptState, jax.Array]:
    """Single gradient-descent step."""
    loss, grads = eqx.filter_value_and_grad(compute_loss)(nn_model)
    updates, new_opt_state = optimizer.update(grads, opt_state, nn_model)
    new_nn = eqx.apply_updates(nn_model, updates)
    return new_nn, new_opt_state, loss


optimizer = optax.adam(3e-3)
opt_state = optimizer.init(eqx.filter(nn, eqx.is_array))

print("Compiling ... (first step takes longer due to JIT)")
losses: list[float] = []
for epoch in range(400):
    nn, opt_state, loss = make_step(nn, opt_state)
    losses.append(float(loss))
    if epoch % 50 == 0:
        print(f"  epoch {epoch:3d}  loss = {float(loss):.5f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(losses, color="royalblue", lw=1.5)
ax.set(xlabel="Epoch", ylabel="MSE loss (log scale)", title="UDE training loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()

## 7 · Results

### 7.1 Trajectory fit

In [ ]:
# Predict with trained UDE
Q_ude = np.array(simulate_ude(nn))[:, 0]
F_ude = 1.0 / (1.0 + Q_ude)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(t_data, Q_true, "k-", lw=2, label="True (hidden forcing)")
axes[0].plot(t_data, Q_ude, "r--", lw=1.8, label="UDE (trained)")
axes[0].plot(t_data, Q_mech, "b:", lw=1.5, label="Mechanistic only")
axes[0].set(ylabel="Quenching state Q")
axes[0].legend()

axes[1].plot(t_data, F_true, "k-", lw=2, label="True")
axes[1].plot(t_data, F_ude, "r--", lw=1.8, label="UDE (trained)")
axes[1].plot(t_data, F_mech, "b:", lw=1.5, label="Mechanistic only")
axes[1].set(xlabel="Time (s)", ylabel="Fluorescence F")
axes[1].legend()

plt.suptitle("UDE vs. mechanistic model vs. true trajectory", y=1.01)
plt.tight_layout()

### 7.2 What did the NDE learn?

We evaluate the trained NDE along the true $Q$ trajectory and compare its output
to the hidden sinusoidal forcing $A\sin(\omega t)$.

In [ ]:
# Evaluate NDE output along the true trajectory
@jax.jit
def eval_nn(t: float, Q: float) -> float:
    I = light_jax(t)
    nn_in = jnp.array([t / T_TOTAL, Q / Q_MAX, I / I_MAX])
    return nn(nn_in)[0]


nde_output = np.array([float(eval_nn(t, Q)) for t, Q in zip(t_data, Q_true)])
true_forcing = A_TRUE * np.sin(OMEGA_TRUE * t_data)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t_data, true_forcing, "k-", lw=2, label=f"True forcing: {A_TRUE}·sin(ωt)")
ax.plot(t_data, nde_output, "r--", lw=1.8, label="NDE output (learned)")
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set(
    xlabel="Time (s)",
    ylabel="dQ/dt contribution",
    title="Recovered sinusoidal forcing vs. true hidden term",
)
ax.legend()
plt.tight_layout()

### 7.3 Residual analysis

In [ ]:
residual_ude = Q_ude - Q_true
residual_mech = Q_mech - Q_true

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_data, residual_mech, "b:", lw=1.5, alpha=0.8, label="Mechanistic only")
ax.plot(t_data, residual_ude, "r--", lw=1.5, alpha=0.8, label="UDE (trained)")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="Time (s)", ylabel="Q_pred − Q_true", title="Prediction residuals")
ax.legend()
plt.tight_layout()

print(f"RMSE mechanistic only : {np.sqrt(np.mean(residual_mech**2)):.4f}")
print(f"RMSE UDE (trained)    : {np.sqrt(np.mean(residual_ude**2)):.4f}")